# 📓 Notebook 03 — Model Training, Tuning & Evaluation
## Student Placement Prediction System
### Authors: Vivek Ramen (LR, DT) · Saket (RF, Tuning) · Yash Bhargava (Evaluation)

This notebook demonstrates:
- Logistic Regression: baseline, L1, L2 regularization sweep, GridSearchCV
- Decision Tree: baseline, depth control, GridSearchCV
- Random Forest: baseline, GridSearchCV, feature importance
- Comprehensive evaluation: confusion matrices, ROC curves, comparison table
- Final model selection and interpretation

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from data_preprocessing import run_full_preprocessing

(X_train, X_test, y_train, y_test,
 preprocessor, df_clean,
 numeric_features, categorical_features, target_col) = run_full_preprocessing()

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print('Data ready ✓')

## SECTION 1: Logistic Regression (Vivek Ramen)

**Working Principle**: Models log-odds of placement as a linear function of features. Sigmoid maps to probability [0,1].

**L2 Regularization**: Penalises large weights uniformly. Prevents any single feature from dominating.

**L1 Regularization**: Encourages sparse solutions — some weights go to exactly 0 (feature selection).

**C = 1/λ**: Smaller C → stronger regularization.

In [ ]:
# ── Baseline LR (L2, C=1) ───────────────────────────────────────────────────
lr_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(penalty='l2', C=1.0, solver='lbfgs',
                                  max_iter=1000, random_state=42))
])
lr_baseline.fit(X_train, y_train)
y_pred_lr = lr_baseline.predict(X_test)

print('Baseline LR (L2, C=1) — Test Set:')
print(classification_report(y_test, y_pred_lr, target_names=['Not Placed', 'Placed']))

In [ ]:
# ── L1 vs L2 Regularization Sweep ───────────────────────────────────────────
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
l1_scores, l2_scores = [], []

for C in C_values:
    # L2
    p = Pipeline([('pp', preprocessor),
                  ('m', LogisticRegression(penalty='l2', C=C, solver='lbfgs',
                                           max_iter=1000, random_state=42))])
    p.fit(X_train, y_train)
    l2_scores.append(f1_score(y_test, p.predict(X_test), zero_division=0))
    
    # L1
    p = Pipeline([('pp', preprocessor),
                  ('m', LogisticRegression(penalty='l1', C=C, solver='saga',
                                           max_iter=2000, random_state=42))])
    p.fit(X_train, y_train)
    l1_scores.append(f1_score(y_test, p.predict(X_test), zero_division=0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(C_values, l2_scores, 'o-', color='#3498DB', linewidth=2, label='L2 (Ridge)')
ax.plot(C_values, l1_scores, 's--', color='#E74C3C', linewidth=2, label='L1 (Lasso)')
ax.set_xscale('log')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('F1-Score (Test Set)')
ax.set_title('Logistic Regression: L1 vs L2 Regularization Effect on F1-Score', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

best_C_l2 = C_values[np.argmax(l2_scores)]
best_C_l1 = C_values[np.argmax(l1_scores)]
print(f'Best C for L2: {best_C_l2}  (F1: {max(l2_scores):.4f})')
print(f'Best C for L1: {best_C_l1}  (F1: {max(l1_scores):.4f})')

In [ ]:
# ── GridSearchCV for LR ──────────────────────────────────────────────────────
param_grid_lr = [
    {'model__penalty': ['l2'], 'model__C': [0.01, 0.1, 1.0, 10.0], 'model__solver': ['lbfgs']},
    {'model__penalty': ['l1'], 'model__C': [0.01, 0.1, 1.0, 10.0], 'model__solver': ['saga']},
]
pipe_lr = Pipeline([('preprocessor', preprocessor),
                     ('model', LogisticRegression(max_iter=2000, random_state=42))])
gs_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='f1', n_jobs=-1, refit=True)
gs_lr.fit(X_train, y_train)
lr_tuned = gs_lr.best_estimator_

print(f'Best LR params: {gs_lr.best_params_}')
print(f'Best CV F1    : {gs_lr.best_score_:.4f}')
print(f'Test F1       : {f1_score(y_test, lr_tuned.predict(X_test)):.4f}')

## SECTION 2: Decision Tree (Vivek Ramen)

**Working Principle**: Recursive partitioning using Gini impurity or entropy. No scaling required.

**Complexity Control**: `max_depth` and `min_samples_leaf` prevent overfitting.

In [ ]:
# ── Baseline DT ──────────────────────────────────────────────────────────────
dt_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])
dt_baseline.fit(X_train, y_train)
print(f'Unconstrained tree depth: {dt_baseline.named_steps["model"].get_depth()}')
print(f'Training accuracy: {dt_baseline.score(X_train, y_train):.4f}')
print(f'Test accuracy    : {dt_baseline.score(X_test, y_test):.4f}')
print('\n(If training >> test, the tree is overfitting)')

In [ ]:
# ── Depth sweep ──────────────────────────────────────────────────────────────
depths = [2, 3, 5, 7, 10, None]
train_scores, test_scores = [], []

for d in depths:
    dt = Pipeline([('pp', preprocessor),
                   ('m', DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, random_state=42))])
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

depth_labels = [str(d) for d in depths]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depth_labels, train_scores, 'o-', color='#3498DB', label='Train Accuracy', linewidth=2)
ax.plot(depth_labels, test_scores,  's-', color='#E74C3C', label='Test Accuracy', linewidth=2)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Depth vs Accuracy (Overfitting Analysis)', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print('Gap between train and test scores shows overfitting.')

In [ ]:
# ── GridSearchCV for DT ──────────────────────────────────────────────────────
param_grid_dt = {
    'model__max_depth'        : [3, 5, 7, 10, None],
    'model__min_samples_split': [2, 10, 20],
    'model__min_samples_leaf' : [1, 5, 10],
    'model__criterion'        : ['gini', 'entropy'],
}
pipe_dt = Pipeline([('preprocessor', preprocessor),
                     ('model', DecisionTreeClassifier(random_state=42))])
gs_dt = GridSearchCV(pipe_dt, param_grid_dt, cv=5, scoring='f1', n_jobs=-1, refit=True)
gs_dt.fit(X_train, y_train)
dt_tuned = gs_dt.best_estimator_

print(f'Best DT params: {gs_dt.best_params_}')
print(f'Best CV F1    : {gs_dt.best_score_:.4f}')
print(f'Test F1       : {f1_score(y_test, dt_tuned.predict(X_test)):.4f}')

## SECTION 3: Random Forest — Ensemble Learning (Saket)

**Ensemble = Bagging + Random Feature Selection + Majority Vote**

Why better than single DT: averaging N de-correlated trees reduces variance.

In [ ]:
# ── Baseline RF ──────────────────────────────────────────────────────────────
rf_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
rf_baseline.fit(X_train, y_train)
print('Baseline RF — Test Set:')
print(classification_report(y_test, rf_baseline.predict(X_test),
                             target_names=['Not Placed', 'Placed']))

In [ ]:
# ── Feature Importance Plot ──────────────────────────────────────────────────
try:
    model = rf_baseline.named_steps['model']
    ct    = rf_baseline.named_steps['preprocessor']
    
    feat_names = []
    for name, trans, cols in ct.transformers_:
        if hasattr(trans, 'get_feature_names_out'):
            feat_names.extend(trans.get_feature_names_out(cols))
        elif hasattr(trans.steps[-1][1], 'get_feature_names_out'):
            feat_names.extend(trans.steps[-1][1].get_feature_names_out(cols))
        else:
            feat_names.extend(cols)
    
    importances = model.feature_importances_
    indices = np.argsort(importances)[-15:]
    
    fig, ax = plt.subplots(figsize=(8, max(4, len(indices) * 0.35)))
    ax.barh([feat_names[i] for i in indices], importances[indices], color='#3498DB')
    ax.set_title('Random Forest — Feature Importances (MDI)\n⚠️ Does NOT imply causality',
                 fontweight='bold')
    ax.set_xlabel('Importance (Mean Decrease in Impurity)')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Feature importance plot error: {e}')

In [ ]:
# ── GridSearchCV for RF (Saket) ──────────────────────────────────────────────
param_grid_rf = {
    'model__n_estimators'    : [50, 100, 200],
    'model__max_depth'       : [None, 5, 10],
    'model__max_features'    : ['sqrt', 'log2'],
    'model__min_samples_leaf': [1, 5],
}
pipe_rf = Pipeline([('preprocessor', preprocessor),
                     ('model', RandomForestClassifier(random_state=42, n_jobs=-1))])
print('Running RF GridSearchCV — this may take a few minutes...')
gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1, verbose=1, refit=True)
gs_rf.fit(X_train, y_train)
rf_tuned = gs_rf.best_estimator_

print(f'\nBest RF params: {gs_rf.best_params_}')
print(f'Best CV F1    : {gs_rf.best_score_:.4f}')
print(f'Test F1       : {f1_score(y_test, rf_tuned.predict(X_test)):.4f}')

## SECTION 4: Comprehensive Evaluation (Yash Bhargava)

**All metrics computed on the held-out test set — seen for the first time here.**

In [ ]:
# ── Compute all metrics ──────────────────────────────────────────────────────
all_models = {
    'LR Baseline': lr_baseline,
    'LR Tuned'   : lr_tuned,
    'DT Baseline': dt_baseline,
    'DT Tuned'   : dt_tuned,
    'RF Baseline': rf_baseline,
    'RF Tuned'   : rf_tuned,
}

rows = []
for name, pipe in all_models.items():
    y_pred = pipe.predict(X_test)
    try:
        y_prob = pipe.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        y_prob = None
        auc = None
    
    rows.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1-Score' : round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC'  : round(auc, 4) if auc else 'N/A',
    })

comparison_df = pd.DataFrame(rows)
print('\n📊 MODEL COMPARISON TABLE (Test Set)')
print('=' * 70)
print(comparison_df.to_string(index=False))

In [ ]:
# ── Confusion Matrices ───────────────────────────────────────────────────────
n = len(all_models)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))

for ax, (name, pipe) in zip(axes, all_models.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Placed','Placed'],
                yticklabels=['Not Placed','Placed'], cbar=False)
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — All Models', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nFP = Predicted Placed, Actually Not Placed → may skip preparation')
print('FN = Predicted Not Placed, Actually Placed → receives unnecessary counselling')

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colours = ['#3498DB', '#2980B9', '#E74C3C', '#C0392B', '#2ECC71', '#27AE60']
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random (AUC=0.50)')

for i, (name, pipe) in enumerate(all_models.items()):
    try:
        y_prob = pipe.predict_proba(X_test)[:,1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, color=colours[i], linewidth=2, label=f'{name} (AUC={auc:.3f})')
    except:
        pass

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('Models with curves further toward top-left have better AUC.')

In [ ]:
# ── Final Model Selection ─────────────────────────────────────────────────────
best_idx = comparison_df['F1-Score'].astype(float).idxmax()
best_model_name = comparison_df.loc[best_idx, 'Model']
best_f1         = comparison_df.loc[best_idx, 'F1-Score']

print(f'\n{'='*60}')
print(f'🏆 SELECTED FINAL MODEL: {best_model_name}')
print(f'   Primary Metric (F1-Score): {best_f1}')
print(f'{'='*60}')
print()
print(comparison_df[comparison_df['Model'] == best_model_name].to_string(index=False))
print()
print('Selection criterion: F1-Score')
print('Rationale: Balances Precision and Recall. Robust to class imbalance.')
print('           Accuracy alone can be misleading if classes are skewed.')

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
import joblib

os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# Save all models
for name, pipe in all_models.items():
    fname = name.lower().replace(' ', '_') + '.pkl'
    joblib.dump(pipe, f'../models/{fname}')
    print(f'  Saved: {fname}')

# Save best as final_model
best_pipe = all_models[best_model_name]
joblib.dump(best_pipe, '../models/final_model.pkl')
print(f'  Saved: final_model.pkl  ({best_model_name})')

# Save comparison CSV
comparison_df.to_csv('../outputs/model_comparison.csv', index=False)
print('  Saved: outputs/model_comparison.csv')